# API Fungsional, Model Multi-Masukan/Keluaran, dan Callback

**Buku:** *Deep Learning dengan Python* — Bab 7, Praktik Terbaik Lanjutan

Keluar dari `Sequential`: membangun model dengan banyak masukan dan keluaran, lalu mengendalikan pelatihan dengan callback.

**Lingkungan:** Google Colab (GPU disarankan), Python 3.10+. Kode bersifat *backend-agnostik* (TensorFlow, JAX, atau PyTorch).

**Perkiraan waktu (GPU T4):** ±5 menit

**Isi notebook:**
- Model sistem tiket dukungan: 3 masukan (judul, isi, tag) → 2 keluaran (prioritas, departemen).
- Menggambar graf model dengan `plot_model` dan mengakses lapisan perantara.
- Callback bawaan: `EarlyStopping`, `ModelCheckpoint`, `ReduceLROnPlateau`, `TensorBoard`.
- Menulis callback sendiri.

In [ ]:
import os
# Pilih backend: "tensorflow", "jax", atau "torch"
os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
print("Keras", keras.__version__, "| backend:", keras.backend.backend())

## 1. Model multi-masukan, multi-keluaran
Datanya acak (sintetis), karena fokusnya adalah **topologi** model, bukan akurasi.

In [ ]:
import numpy as np
from keras import layers

KOSAKATA, N_TAG, N_DEPARTEMEN = 10000, 100, 4

judul = keras.Input(shape=(KOSAKATA,), name="judul")
isi = keras.Input(shape=(KOSAKATA,), name="isi")
tag = keras.Input(shape=(N_TAG,), name="tag")

fitur = layers.Concatenate()([judul, isi, tag])
fitur = layers.Dense(64, activation="relu", name="fitur_bersama")(fitur)
prioritas = layers.Dense(1, activation="sigmoid", name="prioritas")(fitur)
departemen = layers.Dense(N_DEPARTEMEN, activation="softmax", name="departemen")(fitur)

model = keras.Model(inputs=[judul, isi, tag], outputs=[prioritas, departemen])
model.summary()

In [ ]:
keras.utils.plot_model(model, "tiket.png", show_shapes=True)

In [ ]:
N = 1280
rng = np.random.default_rng(0)
data_judul = rng.integers(0, 2, size=(N, KOSAKATA)).astype("float32")
data_isi = rng.integers(0, 2, size=(N, KOSAKATA)).astype("float32")
data_tag = rng.integers(0, 2, size=(N, N_TAG)).astype("float32")
y_prioritas = rng.random((N, 1)).astype("float32")
y_departemen = rng.integers(0, N_DEPARTEMEN, size=(N,))

# Loss dan metrik diberikan per keluaran (dipetakan lewat nama lapisan)
model.compile(optimizer="rmsprop",
              loss={"prioritas": "mean_squared_error", "departemen": "sparse_categorical_crossentropy"},
              loss_weights={"prioritas": 1.0, "departemen": 0.5},
              metrics={"prioritas": ["mae"], "departemen": ["accuracy"]})
model.fit({"judul": data_judul, "isi": data_isi, "tag": data_tag},
          {"prioritas": y_prioritas, "departemen": y_departemen},
          epochs=2, validation_split=0.2)

### Memakai ulang lapisan perantara
Graf fungsional bisa "dibedah": di sini kita menambah keluaran ketiga tanpa melatih ulang dari nol.

In [ ]:
fitur_tengah = model.get_layer("fitur_bersama").output
print(fitur_tengah.shape)
kesulitan = layers.Dense(3, activation="softmax", name="kesulitan")(fitur_tengah)
model_baru = keras.Model(model.inputs, [prioritas, departemen, kesulitan])
keras.utils.plot_model(model_baru, "tiket_baru.png", show_shapes=True)

## 2. Callback pada MNIST

In [ ]:
(x_latih, y_latih), (x_uji, y_uji) = keras.datasets.mnist.load_data()
x_latih = x_latih.reshape(-1, 784).astype("float32") / 255
x_uji = x_uji.reshape(-1, 784).astype("float32") / 255
x_latih, x_val = x_latih[10000:], x_latih[:10000]
y_latih, y_val = y_latih[10000:], y_latih[:10000]

def model_mnist():
    m = keras.Sequential([keras.Input(shape=(784,)),
                          layers.Dense(512, activation="relu"), layers.Dropout(0.5),
                          layers.Dense(10, activation="softmax")])
    m.compile(optimizer="rmsprop", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

In [ ]:
import matplotlib.pyplot as plt

class LossPerBatch(keras.callbacks.Callback):
    """Callback buatan sendiri: mencatat loss tiap batch dan menggambarnya di akhir epoch."""
    def on_train_begin(self, logs=None):
        self.loss = []
    def on_train_batch_end(self, batch, logs=None):
        self.loss.append(logs["loss"])
    def on_train_end(self, logs=None):
        plt.figure(figsize=(8, 3)); plt.plot(self.loss)
        plt.xlabel("Batch"); plt.ylabel("Loss"); plt.title("Loss per batch"); plt.show()

daftar_callback = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint("mnist_terbaik.keras", monitor="val_loss", save_best_only=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
    keras.callbacks.TensorBoard(log_dir="log_tensorboard"),
    LossPerBatch(),
]
model = model_mnist()
riwayat = model.fit(x_latih, y_latih, epochs=30, batch_size=128,
                    validation_data=(x_val, y_val), callbacks=daftar_callback)
print("Berhenti setelah", len(riwayat.history["loss"]), "epoch")
print(f"Akurasi uji: {model.evaluate(x_uji, y_uji, verbose=0)[1]:.2%}")

### TensorBoard
Jalankan sel di bawah untuk membuka dasbor TensorBoard di dalam Colab.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir log_tensorboard

**Eksperimen:** ubah `patience` pada `EarlyStopping` menjadi 1 atau 10. Apa pengaruhnya pada jumlah epoch dan akurasi uji?